In [ ]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import os
from torch.utils.data import DataLoader
from src.data.dataset import FundusDataset
from transformers import ViTForImageClassification
from transformers import ViTForImageClassification, ViTImageProcessor

In [ ]:
def cpu():
    if torch.cuda.is_available():
        return 'cuda'
    else:
        return 'cpu'

result = cpu()
print(result)
device = torch.device(result)
print(device)

In [ ]:
train_dir = r'F:\graduation_project\data\processed\Training Images'
train_excel_dir = r'F:\graduation_project\data\processed\training annotation (English).xlsx'

# 训练集
train_dataset = FundusDataset(train_dir , train_excel_dir , is_training=True)

In [ ]:
# 创建DataLoader
train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    drop_last=True
)

In [ ]:
# 获取一个batch的真实数据
data_iter = iter(train_loader)
images, true_labels, img_names = next(data_iter)

In [ ]:
# 用已经预训练过的模型可以增加准确率
local_model_path = r'C:/Users/lenovo/Desktop/graduation_project/models/vit-base-patch16-224'


# 加载预训练的处理器
processor = ViTImageProcessor.from_pretrained(local_model_path)

# 加载预训练的ViT模型
model = ViTForImageClassification.from_pretrained(
    local_model_path,
    num_labels=8,  # 指定分类数
    ignore_mismatched_sizes=True  # 允许模型权重尺寸不匹配时自动调整
)
model = model.to(device)

In [ ]:
with torch.no_grad():
    images = images.to(device)
    outputs = model(images)
    logits = outputs.logits.cpu()
    probs = torch.sigmoid(logits)

print(f"\n模型输出 (logits):")
print(logits)

print(f"\nSigmoid后 (概率):")
print(probs)

# ========== 计算损失 ==========
print("\n" + "="*60)
print("损失函数计算过程")
print("="*60)

# BCEWithLogitsLoss
criterion = nn.BCEWithLogitsLoss()
total_loss = criterion(logits, true_labels)

print(f"\n总损失 (BCEWithLogitsLoss): {total_loss.item():.6f}")